# GPU Pipeline Runner — Google Colab

Runs the full surveillance pipeline (YOLO + DeepSORT + CLIP) and the UCF-Crime evaluation on Colab's free T4 GPU.

**Speedup vs CPU laptop:** 1-hour video ≈ 2–4 min (vs ~15–30 min); CLIP encoding ~20× faster.

**Workflow:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run the cells top to bottom
3. Download the produced `cache_artifacts.zip` — extract it into the project's `.cache/` folder on your laptop and the local Streamlit UI will get an instant cache hit for the same video (cache keys are content-based, machine-independent).

In [ ]:
# 1. Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Clone the repo and install dependencies (~3 min)
!git clone https://github.com/jaisris/smart_query_driven_surveillance_vlm.git
%cd smart_query_driven_surveillance_vlm
!pip install -q -r requirements.txt

In [ ]:
# 3. Mount Google Drive — put your video in Drive first (e.g. MyDrive/surveillance/)
from google.colab import drive
drive.mount('/content/drive')

VIDEO_PATH = '/content/drive/MyDrive/surveillance/my_video.mp4'  # <-- EDIT THIS
import os
assert os.path.exists(VIDEO_PATH), f'Video not found: {VIDEO_PATH}'
print(f'Video found: {os.path.getsize(VIDEO_PATH)/1e6:.0f} MB')

In [ ]:
# 4. Run the full pipeline on GPU (device='auto' picks CUDA automatically)
import os, sys, time, hashlib
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
sys.path.insert(0, '.')

from pipeline.video_pipeline import VideoPipeline

def hash_file(path, chunk_mb=8):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(chunk_mb * 1024 * 1024):
            h.update(chunk)
    return h.hexdigest()

content_hash = hash_file(VIDEO_PATH)
t0 = time.time()
result = VideoPipeline().run(VIDEO_PATH, content_hash=content_hash)
print(f'\nPipeline done in {time.time()-t0:.1f}s')
print(f'Frames indexed: {len(result.frame_index_entries)}')
print(f'Tracks: {len(result.track_histories)}   Anomalies: {len(result.anomaly_events)}')

In [ ]:
# 5. Try natural language queries against the indexed video
from retrieval.query_encoder import QueryEncoder
from retrieval.similarity_search import SimilaritySearch
from retrieval.temporal_localizer import localize_segments

search = SimilaritySearch()
search.build_index(result.embedding_matrix, result.frame_index_entries)
qenc = QueryEncoder()

for query in ['a person walking', 'a car on the road', 'people talking']:
    vec = qenc.encode(query)
    segments = localize_segments(search.search(vec, top_k=20))
    print(f'\nQuery: "{query}"')
    for seg in segments[:3]:
        print(f'  {seg.start_sec:7.1f}s – {seg.end_sec:7.1f}s   score={seg.peak_score:.3f}')

In [ ]:
# 6. (Optional) UCF-Crime evaluation on GPU — full dataset feasible here.
# First upload/extract the Kaggle dataset (odins0n/ucf-crime-dataset) to Drive,
# then point --data-root at its Test folder.
UCF_TEST = '/content/drive/MyDrive/surveillance/archive/Test'  # <-- EDIT IF USING
if os.path.isdir(UCF_TEST):
    !python evaluation/run_ucf_eval.py --data-root "$UCF_TEST" --frames-per-class 2000 --batch-size 256
else:
    print('UCF dataset not found in Drive — skipping (edit UCF_TEST above to enable)')

In [ ]:
# 7. Zip the cache + results for download → extract into project .cache/ locally
!zip -r cache_artifacts.zip .cache Docs/ucf_eval_results.json Docs/ucf_roc_curve.png 2>/dev/null
from google.colab import files
files.download('cache_artifacts.zip')